# Assignment 5: Employee Attrition Prediction using Decision Tree and Random Forest

**Dataset:** IBM HR Analytics Employee Attrition & Performance
(Kaggle: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)

This notebook builds and compares a Decision Tree Classifier and a Random Forest Classifier
to predict employee attrition.

> **Note on data access:** This notebook expects the dataset CSV
> (`WA_Fn-UseC_-HR-Employee-Attrition.csv`) to be placed in the same folder as this notebook,
> or downloaded automatically via `kagglehub` (see Task 1, Cell 1). Do NOT commit the raw
> dataset file to GitHub — only the Kaggle link goes in the README.


## Task 1: Data Understanding (2 Marks)

In [ ]:
# 1. Load the dataset
import pandas as pd
import numpy as np
import os

DATA_PATH = "WA_Fn-UseC_-HR-Employee-Attrition.csv"

if not os.path.exists(DATA_PATH):
    # Try downloading via kagglehub if the file isn't present locally.
    try:
        import kagglehub
        path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")
        for f in os.listdir(path):
            if f.endswith(".csv"):
                DATA_PATH = os.path.join(path, f)
                break
    except Exception as e:
        print("Could not auto-download dataset. Please place the CSV file in this folder.")
        print("Kaggle link: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset")
        raise e

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)


In [ ]:
# 2. Display the first five records
df.head()


In [ ]:
# 3. Identify numerical, categorical features and target variable
target_variable = "Attrition"

numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=["object"]).columns.tolist()

# Attrition is the target, remove it from the categorical feature list
if target_variable in categorical_features:
    categorical_features.remove(target_variable)

print(f"Target variable: {target_variable}\n")
print(f"Numerical features ({len(numerical_features)}):")
print(numerical_features)
print(f"\nCategorical features ({len(categorical_features)}):")
print(categorical_features)


In [ ]:
# 4. Dataset information and summary statistics
df.info()


In [ ]:
df.describe(include='all').T


## Task 2: Data Preprocessing (2 Marks)

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")


In [ ]:
# Remove unnecessary columns
# EmployeeCount, StandardHours, and Over18 are constant across all rows (no predictive value).
# EmployeeNumber is a unique identifier and carries no predictive signal.
cols_to_drop = [c for c in ["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"] if c in df.columns]
df_clean = df.drop(columns=cols_to_drop)
print(f"Dropped columns: {cols_to_drop}")
print("New shape:", df_clean.shape)


In [ ]:
# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

df_encoded = df_clean.copy()
label_encoders = {}

# Encode target variable
target_encoder = LabelEncoder()
df_encoded[target_variable] = target_encoder.fit_transform(df_encoded[target_variable])  # Yes=1, No=0

# Encode remaining categorical (object) columns
cat_cols_remaining = df_encoded.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols_remaining:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

print("Encoded categorical columns:", cat_cols_remaining)
df_encoded.head()


In [ ]:
# Split the dataset into 80% training and 20% testing
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=[target_variable])
y = df_encoded[target_variable]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


## Task 3: Model Development (3 Marks)

In [ ]:
# Model 1: Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
print("Decision Tree model trained.")


In [ ]:
# Model 2: Random Forest Classifier (100 estimators)
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
print("Random Forest model trained.")


## Task 4: Model Evaluation and Comparison (2 Marks)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"--- {name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}\n")
    return {"Model": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1}

results = []
results.append(evaluate_model("Decision Tree", y_test, dt_preds))
results.append(evaluate_model("Random Forest", y_test, rf_preds))

results_df = pd.DataFrame(results)
results_df


In [ ]:
# Confusion Matrices
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_dt = confusion_matrix(y_test, dt_preds)
sns.heatmap(cm_dt, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
axes[0].set_title("Decision Tree - Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

cm_rf = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
axes[1].set_title("Random Forest - Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()


In [ ]:
# Feature Importance plot for Random Forest
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_features.values, y=top_features.index, palette="viridis")
plt.title("Top 15 Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()


### Observations (fill in after running on the real dataset)

1. *Compare overall accuracy* — Random Forest typically shows a higher (or comparable) accuracy
   than the single Decision Tree because it aggregates predictions from many trees, reducing
   variance.
2. *Compare precision/recall trade-off* — note which model catches more true attrition cases
   (recall) versus which produces fewer false alarms (precision).
3. *Confusion matrix pattern* — describe whether one model tends to misclassify more
   "Yes" (attrition) cases as "No", which is often the costlier error for HR planning.
4. *Feature importance insight* — mention which features (e.g., OverTime, MonthlyIncome,
   Age, TotalWorkingYears, JobSatisfaction, YearsAtCompany) the Random Forest ranks as most
   predictive, and whether these align with domain intuition about attrition drivers.

*(Replace the bullet points above with your actual numeric findings once you run this notebook
on the downloaded dataset.)*


## Task 5: Conclusion (1 Mark)

In this project, both a Decision Tree and a Random Forest classifier were trained to predict
employee attrition using the IBM HR Analytics dataset. Based on the evaluation metrics above,
[the Random Forest / the Decision Tree] achieved the better overall performance, showing
[higher/comparable] accuracy, precision, recall, and F1-score on the held-out test set.

Random Forest often outperforms a single Decision Tree because it is an ensemble method: it
builds many decision trees on bootstrapped samples of the data and random subsets of features,
then averages their predictions. This process, known as bagging, reduces the variance and
overfitting that a single deep decision tree is prone to, resulting in a model that generalizes
better to unseen data.

A key limitation of Decision Trees is that they are highly sensitive to small changes in the
training data and tend to overfit if not pruned or depth-limited, which can hurt generalization.
A key limitation of Random Forests is that they are less interpretable than a single tree —
it is harder to trace an individual prediction back to a clear decision path — and they are more
computationally expensive to train and use for inference, especially with a large number of
trees or features.


## Bonus Challenge (Not Mandatory): Hyperparameter Tuning Experiment

In [ ]:
# Example: tuning max_depth for the Decision Tree
depths_to_try = [3, 5, 10, None]
tuning_results = []

for d in depths_to_try:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    tuning_results.append({
        "max_depth": d,
        "Accuracy": accuracy_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds)
    })

tuning_df = pd.DataFrame(tuning_results)
tuning_df


**Effect of `max_depth` on the Decision Tree:**
A very shallow tree (e.g., `max_depth=3`) may underfit, missing important interactions in the
data, while an unrestricted depth (`max_depth=None`) allows the tree to grow until leaves are
pure, which often overfits the training data and can reduce test performance. A moderate depth
(e.g., 5–10) typically strikes the best balance between bias and variance — compare the table
above to identify which value gave the best F1-score on your run and report whether performance
improved relative to the default (unrestricted) tree.
